In [1]:
# Clear variables
%reset -f

import duckdb
import pandas as pd

# Constants
DB_PATH = "/home/rud/Documents/01_IE/08_INTR/03_Material_Analysis/project/industrial_cluster.db"
COMPANY_ID = "C010"  # Change this

# Persistent connection with SQLite attached
con = duckdb.connect()
con.execute(f"ATTACH '{DB_PATH}' AS db (TYPE SQLITE)")

In [2]:
# ── CELL 2: Company header ─────────────────────────────────────────────────────

df_header = con.execute("""
    SELECT
        c.company_id,
        c.name                      AS company_name,
        c.normalize_stream_id       AS reference_stream_id,
        ref.stream_name             AS reference_stream_name,
        ref.flow_kton_per_year      AS reference_flow_kton_per_year
    FROM db.companies c
    LEFT JOIN db.streams ref ON c.normalize_stream_id = ref.stream_id
    WHERE c.company_id = $company_id
""", {"company_id": COMPANY_ID}).df()

display(df_header)

,company_id,company_name,reference_stream_id,reference_stream_name,reference_flow_kton_per_year
0,C010,Methanol Production Plant,S131,PS-CH3OH,93.716777


In [3]:
# ── CELL 3: Stream-level mass balance ─────────────────────────────────────────

df_streams = con.execute("""
    SELECT
        s.stream_id,
        s.stream_name,
        s.stream_type,
        s.direction,
        s.flow_kton_per_year,
        s.norm_flow_kton_per_year,
        ROUND(s.carbon_pct * 100, 2)                        AS carbon_pct_display,
        s.carbon_pct_complete,
        ROUND(s.flow_kton_per_year * s.carbon_pct, 3)       AS carbon_flow_kton_per_year,
        s.temperature_c,
        s.pressure_bar,
        s.notes
    FROM db.streams s
    WHERE s.company_id = $company_id
    ORDER BY
        CASE s.direction  WHEN 'input' THEN 0 ELSE 1 END,
        CASE s.stream_type
            WHEN 'raw_material' THEN 0
            WHEN 'product'      THEN 1
            WHEN 'waste'        THEN 2
            ELSE 3
        END,
        s.flow_kton_per_year DESC
""", {"company_id": COMPANY_ID}).df()

df_streams

,stream_id,stream_name,stream_type,direction,flow_kton_per_year,norm_flow_kton_per_year,carbon_pct_display,carbon_pct_complete,carbon_flow_kton_per_year,temperature_c,pressure_bar,notes
0,S125,FS-AIR,raw_material,input,827.760128,2649.771426,0.00,1,0.000,20.000000,1.02,None
1,S130,FS-WATER,raw_material,input,157.213547,503.261695,0.00,1,0.000,20.000000,1.02,None
2,S126,FS-BFW1,raw_material,input,80.204490,256.745352,0.00,1,0.000,204.346000,17.00,None
3,S127,FS-Methane,raw_material,input,50.000000,160.056720,74.88,1,37.441,20.000000,1.02,None
4,S129,FS-CO2,raw_material,input,38.817403,124.259726,27.29,1,10.594,20.000000,1.02,None
5,S128,FS-Methane-2,raw_material,input,32.220319,103.141571,74.88,1,24.127,20.000000,1.02,None
6,S133,WS-FG,waste,output,859.980447,2752.912998,5.42,1,46.602,64.218340,1.02,None
7,S136,WS-W1,waste,output,88.970389,284.806175,0.00,1,0.000,40.000000,1.02,None
8,S135,WS-PURGE,waste,output,33.801060,108.201737,31.71,1,10.718,40.359253,40.00,None
9,S137,WS-W2,waste,output,22.448779,71.861559,0.16,1,0.035,40.000000,1.02,None


In [4]:
# ── CELL 4: Totals by direction + stream_type (closure check) ─────────────────

df_totals = con.execute("""
    SELECT * FROM (
        SELECT
            direction,
            stream_type,
            COUNT(*)                                            AS stream_count,
            ROUND(SUM(flow_kton_per_year), 3)                  AS total_flow_kton_per_year,
            ROUND(SUM(norm_flow_kton_per_year), 3)             AS total_norm_flow,
            ROUND(SUM(flow_kton_per_year * carbon_pct), 3)     AS total_carbon_flow_kton_per_year
        FROM db.streams
        WHERE company_id = $company_id
          AND flow_kton_per_year IS NOT NULL
        GROUP BY direction, stream_type

        UNION ALL

        SELECT
            direction,
            'TOTAL'                                             AS stream_type,
            COUNT(*)                                            AS stream_count,
            ROUND(SUM(flow_kton_per_year), 3)                  AS total_flow_kton_per_year,
            ROUND(SUM(norm_flow_kton_per_year), 3)             AS total_norm_flow,
            ROUND(SUM(flow_kton_per_year * carbon_pct), 3)     AS total_carbon_flow_kton_per_year
        FROM db.streams
        WHERE company_id = $company_id
          AND flow_kton_per_year IS NOT NULL
        GROUP BY direction
    )
    ORDER BY
        CASE direction   WHEN 'input' THEN 0 ELSE 1 END,
        CASE stream_type WHEN 'TOTAL' THEN 99 ELSE 0 END
""", {"company_id": COMPANY_ID}).df()

df_totals

,direction,stream_type,stream_count,total_flow_kton_per_year,total_norm_flow,total_carbon_flow_kton_per_year
0,input,raw_material,6,1186.216,3797.236,72.162
1,input,TOTAL,6,1186.216,3797.236,72.162
2,output,products,2,173.921,556.745,35.128
3,output,waste,5,1012.295,3240.492,59.315
4,output,TOTAL,7,1186.216,3797.237,94.443


In [5]:
# ── CELL 6: Mass balance closure summary ──────────────────────────────────────

totals = (
    df_totals[df_totals["stream_type"] == "TOTAL"]
    .set_index("direction")
)

total_in  = totals.loc["input",  "total_flow_kton_per_year"]
total_out = totals.loc["output", "total_flow_kton_per_year"]
gap       = total_in - total_out
closure   = (1 - abs(gap) / total_in) * 100 if total_in else float("nan")

carbon_in  = totals.loc["input",  "total_carbon_flow_kton_per_year"]
carbon_out = totals.loc["output", "total_carbon_flow_kton_per_year"]
carbon_gap = carbon_in - carbon_out

print(f"Company : {COMPANY_ID}")
print(f"{'─'*40}")
print(f"Total input       : {total_in:>10.3f}  kton/yr")
print(f"Total output      : {total_out:>10.3f}  kton/yr")
print(f"Gap (in − out)    : {gap:>10.3f}  kton/yr")
print(f"Closure           : {closure:>10.2f}  %")
print(f"{'─'*40}")
print(f"Carbon input      : {carbon_in:>10.3f}  kton C/yr")
print(f"Carbon output     : {carbon_out:>10.3f}  kton C/yr")
print(f"Carbon gap        : {carbon_gap:>10.3f}  kton C/yr")

Company : C010
────────────────────────────────────────
Total input       :   1186.216  kton/yr
Total output      :   1186.216  kton/yr
Gap (in − out)    :      0.000  kton/yr
Closure           :     100.00  %
────────────────────────────────────────
Carbon input      :     72.162  kton C/yr
Carbon output     :     94.443  kton C/yr
Carbon gap        :    -22.281  kton C/yr
